# Insurance Fraud Claims Detection Engine
### Machine Learning Prototype — Project Report

---

| Field | Detail |
|---|---|
| **Project** | Insurance Fraud Claims Detection |
| **Domain** | FinTech / InsurTech |
| **Models** | Logistic Regression (Baseline) + XGBoost |
| **Explainability** | SHAP (SHapley Additive exPlanations) |
| **Deployment** | FastAPI + Jinja2 + Render.com |
| **Live URL** | https://insurance-fraud-engine.onrender.com |
| **Dataset** | Auto Insurance Claims — 1,000 records, 39 features |


## 1. Problem Statement

Insurance fraud costs the U.S. industry **$40 billion per year** (FBI estimate), adding ~\$400–700 to average household premiums.  
Auto insurance claims fraud — staged accidents, inflated damage, false injury claims — accounts for a significant share.

**Goal:** Build a binary classifier that flags fraudulent auto insurance claims using policyholder and incident features, with full model explainability so adjusters understand *why* a claim is flagged.

**Target variable:** `fraud_reported` (Y = fraud, N = legitimate)

**Success metrics:** ROC-AUC ≥ 0.80, Recall ≥ 0.70 (catch most fraud), Precision ≥ 0.60 (limit false positives)

## 2. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys, os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_score, recall_score, f1_score,
                             confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
import shap

# Point to project root
ROOT = Path().resolve()
sys.path.insert(0, str(ROOT))

from src.data_loader import load_data, split_data
from src.feature_engineering import FeatureEngineer
from src.models.baseline import BaselineModel
from src.models.xgboost_model import XGBoostModel
from src import evaluator

print('All imports successful.')
print(f'pandas {pd.__version__} | numpy {np.__version__}')

## 3. Data Loading & Exploratory Data Analysis

In [ ]:
DATA_PATH = ROOT / 'data' / 'insurance_claims.csv'
df = load_data(str(DATA_PATH))

print(f'Dataset shape: {df.shape}')
print(f'Fraud rate: {df["fraud_reported"].mean()*100:.1f}%')
df.head()

In [ ]:
# Class distribution
fraud_counts = df['fraud_reported'].value_counts()
labels = ['Legitimate (0)', 'Fraudulent (1)']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
axes[0].pie(fraud_counts, labels=labels, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Fraud vs Legitimate Claims', fontsize=13, fontweight='bold')

# Bar chart
axes[1].bar(labels, fraud_counts.values, color=['#4CAF50', '#F44336'], edgecolor='white')
axes[1].set_ylabel('Count')
axes[1].set_title('Class Distribution', fontsize=13, fontweight='bold')
for i, v in enumerate(fraud_counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Legitimate: {fraud_counts[0]} | Fraudulent: {fraud_counts[1]}')
print(f'Class imbalance ratio: {fraud_counts[0]/fraud_counts[1]:.1f}:1')

In [ ]:
# Total claim amount distribution by fraud label
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for label, color in [(0, '#4CAF50'), (1, '#F44336')]:
    subset = df[df['fraud_reported'] == label]['total_claim_amount']
    axes[0].hist(subset, bins=30, alpha=0.6, color=color,
                 label='Legitimate' if label == 0 else 'Fraud')
axes[0].set_xlabel('Total Claim Amount ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Claim Amount Distribution by Label', fontweight='bold')
axes[0].legend()

# Incident type by fraud
inc_fraud = df.groupby('incident_type')['fraud_reported'].mean().sort_values(ascending=False)
axes[1].barh(inc_fraud.index, inc_fraud.values * 100, color='#E91E63')
axes[1].set_xlabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Incident Type', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_charts.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Dataset statistics summary
print('=== DATASET SUMMARY ===')
print(f'Total records    : {len(df):,}')
print(f'Total features   : {df.shape[1]}')
print(f'Fraudulent claims: {df["fraud_reported"].sum()} ({df["fraud_reported"].mean()*100:.1f}%)')
print(f'Legitimate claims: {(df["fraud_reported"]==0).sum()} ({(1-df["fraud_reported"].mean())*100:.1f}%)')
print(f'Missing values   : {df.isnull().sum().sum()}')
print()
print('Numeric feature statistics:')
df.describe().round(2)

## 4. Feature Engineering

Key transformations applied:
- **Label encoding** of categorical columns
- **Median/mode imputation** for missing values  
- **Derived features:**
  - `claim_to_premium_ratio` = total_claim_amount / policy_annual_premium
  - `is_night_incident` = 1 if incident hour < 6 or > 22
  - `no_police_no_witness` = 1 if witnesses=0 AND no police report
- **StandardScaler** for Logistic Regression; raw for XGBoost

In [ ]:
# Split data: 70% train / 15% val / 15% test
train_df, val_df, test_df = split_data(df)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# Feature engineer: two instances (scaled for LR, unscaled for XGBoost)
fe_baseline = FeatureEngineer()
fe_xgb = FeatureEngineer()

X_train_sc, y_train = fe_baseline.fit_transform_scaled(train_df)
X_val_sc, y_val     = fe_baseline.transform_scaled(val_df)
X_test_sc, y_test   = fe_baseline.transform_scaled(test_df)

X_train_enc, _      = fe_xgb.fit_transform(train_df)
X_val_enc, _        = fe_xgb.transform(val_df)
X_test_enc, _       = fe_xgb.transform(test_df)

print(f'Feature count: {X_train_sc.shape[1]}')
print(f'Class balance in train: {y_train.mean()*100:.1f}% fraud')

In [ ]:
# Show derived features
print('Derived features added:')
derived = ['claim_to_premium_ratio', 'is_night_incident', 'no_police_no_witness']
for col in derived:
    cols = [c for c in X_train_enc.columns if col in c]
    if cols:
        print(f'  {cols[0]}: mean={X_train_enc[cols[0]].mean():.3f}')

## 5. Model Training

### 5.1 Baseline — Logistic Regression
- `class_weight='balanced'` to handle imbalance
- `max_iter=1000` for convergence

In [ ]:
baseline = BaselineModel()
baseline.train(X_train_sc, y_train)

lr_proba = baseline.predict_proba(X_val_sc)
lr_pred  = baseline.predict(X_val_sc)

lr_metrics = evaluator.compute_metrics(y_val, lr_proba)
print('Logistic Regression — Validation Metrics')
print('=' * 40)
for k, v in lr_metrics.items():
    print(f'  {k:<12}: {v:.4f}')

### 5.2 XGBoost Classifier
- `scale_pos_weight=3.0` for class imbalance
- GridSearchCV: n_estimators, max_depth, learning_rate

In [ ]:
xgb_model = XGBoostModel(scale_pos_weight=3.0)
xgb_model.train(X_train_enc, y_train)

xgb_proba = xgb_model.predict_proba(X_val_enc)
xgb_pred  = xgb_model.predict(X_val_enc)

xgb_metrics = evaluator.compute_metrics(y_val, xgb_proba)
print('XGBoost — Validation Metrics')
print('=' * 40)
for k, v in xgb_metrics.items():
    print(f'  {k:<12}: {v:.4f}')

print()
print('Best XGBoost params:', xgb_model.model.best_params_)

## 6. Model Evaluation & Comparison

In [ ]:
# Side-by-side comparison table
comparison = pd.DataFrame({
    'Logistic Regression': lr_metrics,
    'XGBoost': xgb_metrics
}).T.round(4)
comparison.index.name = 'Model'
print('Model Comparison — Validation Set')
comparison

In [ ]:
# ROC Curves
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
for name, proba, color in [
    ('Logistic Regression', lr_proba, '#2196F3'),
    ('XGBoost', xgb_proba, '#FF5722')
]:
    fpr, tpr, _ = roc_curve(y_val, proba)
    auc = roc_auc_score(y_val, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves', fontweight='bold'); axes[0].legend()
axes[0].grid(alpha=0.3)

# Precision-Recall
for name, proba, color in [
    ('Logistic Regression', lr_proba, '#2196F3'),
    ('XGBoost', xgb_proba, '#FF5722')
]:
    prec, rec, _ = precision_recall_curve(y_val, proba)
    ap = average_precision_score(y_val, proba)
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.3f})', color=color, lw=2)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves', fontweight='bold'); axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, name, pred in [
    (axes[0], 'Logistic Regression', lr_pred),
    (axes[1], 'XGBoost', xgb_pred)
]:
    cm = confusion_matrix(y_val, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    ax.set_title(f'{name}\nConfusion Matrix', fontweight='bold')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Test set final evaluation (XGBoost)
test_proba = xgb_model.predict_proba(X_test_enc)
test_metrics = evaluator.compute_metrics(y_test, test_proba)

print('XGBoost — FINAL TEST SET Metrics')
print('=' * 40)
for k, v in test_metrics.items():
    print(f'  {k:<12}: {v:.4f}')

## 7. SHAP Explainability

SHAP (SHapley Additive exPlanations) provides:
- **Global view (beeswarm):** Which features matter most across all claims
- **Local view (waterfall):** Why a specific claim was flagged

In [ ]:
# Global SHAP — beeswarm plot
best_model = xgb_model.model.best_estimator_
explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_val_enc)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_val_enc, max_display=15, show=False)
plt.title('SHAP Feature Importance (Global)', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()
print('Top features driving fraud predictions shown above.')

In [ ]:
# Local SHAP — waterfall for a high-risk claim
high_risk_idx = np.where(xgb_proba > 0.7)[0]
if len(high_risk_idx) == 0:
    high_risk_idx = [np.argmax(xgb_proba)]

claim_idx = high_risk_idx[0]
X_row = X_val_enc.iloc[[claim_idx]]

sv = explainer(X_row)
plt.figure(figsize=(10, 5))
shap.plots.waterfall(sv[0], max_display=12, show=False)
plt.title(f'SHAP Waterfall — Claim #{claim_idx} (Fraud Prob: {xgb_proba[claim_idx]:.1%})',
          fontweight='bold')
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Claim #{claim_idx}: Fraud probability = {xgb_proba[claim_idx]:.1%}')
print(f'Actual label: {"FRAUD" if y_val.iloc[claim_idx]==1 else "LEGITIMATE"}')

## 8. System Architecture

```
insurance-fraud-engine/
├── data/
│   └── insurance_claims.csv      # 1,000 rows, 39 features
├── src/
│   ├── data_loader.py            # Load, clean, stratified split
│   ├── feature_engineering.py    # Encode, scale, derived features
│   ├── evaluator.py              # Metrics + matplotlib charts
│   ├── models/
│   │   ├── baseline.py           # Logistic Regression
│   │   └── xgboost_model.py      # XGBoost + GridSearchCV
│   └── explainer/
│       └── shap_explainer.py     # SHAP beeswarm + waterfall
├── web/
│   ├── main.py                   # FastAPI — 4 routes
│   └── templates/                # Jinja2 + Tailwind CSS
│       ├── base.html
│       ├── overview.html         # Dataset stats + charts
│       ├── evaluation.html       # ROC/PR/metrics table
│       ├── explainability.html   # SHAP global + local
│       └── scorer.html           # Manual claim scorer
├── tests/                        # 28 pytest tests (21 ML + 7 web)
├── Dockerfile                    # Docker build for Render
└── render.yaml                   # Render deployment config
```

**Web App Pages:**
| Page | Route | Description |
|---|---|---|
| Overview | `/` | Dataset stats, pie chart, histogram, sample table |
| Evaluation | `/evaluation` | ROC, PR curves, confusion matrix, metrics table |
| Explainability | `/explainability` | SHAP beeswarm (global) + waterfall (per claim) |
| Claim Scorer | `/scorer` | Manual input form → fraud probability + SHAP |


## 9. Key Findings & Conclusions

### Model Performance Summary

| Metric | Logistic Regression | XGBoost | Target |
|---|---|---|---|
| ROC-AUC | ~0.78 | ~0.87 | ≥ 0.80 |
| Recall | ~0.73 | ~0.76 | ≥ 0.70 |
| Precision | ~0.55 | ~0.63 | ≥ 0.60 |
| F1-Score | ~0.63 | ~0.69 | — |

XGBoost meets all three targets; Logistic Regression meets Recall but falls short on Precision.

### Top Fraud Indicators (from SHAP)
1. **`claim_to_premium_ratio`** — high claim relative to premium is the strongest fraud signal
2. **`no_police_no_witness`** — no official verification strongly predicts fraud
3. **`is_night_incident`** — late-night incidents (10 PM–6 AM) correlate with fraud
4. **`total_claim_amount`** — inflated claims are a classic fraud pattern
5. **`incident_severity`** — major damage claims have higher fraud rates

### Business Value
- Flags **~76% of actual fraud** (recall) for human review
- **63% precision** means 3 in 5 flagged claims are genuinely suspicious
- SHAP waterfall gives adjusters an explainable reason to investigate each flagged claim
- Fully deployed as a web app — any adjuster can score a claim in real time

### Future Improvements
- Add network graph features (insured ↔ attorney ↔ body shop links)
- Experiment with LightGBM and ensemble stacking
- Add claim text analysis (NLP on incident descriptions)
- Real-time API integration with claims management systems

In [ ]:
print('=' * 50)
print('PROJECT COMPLETE')
print('=' * 50)
print(f'Live URL : https://insurance-fraud-engine.onrender.com')
print(f'GitHub   : https://github.com/appalareddy17/insurance-fraud-engine')
print()
print('Test set — Final XGBoost Results:')
for k, v in test_metrics.items():
    print(f'  {k:<12}: {v:.4f}')